In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML"
)

ML_FILE = (
    PROJECT_ROOT
    / "data"
    / "ml_ready"
    / "thermal_context_ml_ready_prototype.csv"
)

data = pd.read_csv(ML_FILE)

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (570717, 32)


,acq_date,start_time,end_time,latitude,longitude,observation_count,duration_minutes,mean_frp,peak_frp,total_frp,...,lat_cell,lon_cell,date_only,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected,anomaly_score,is_anomaly
0,2025-01-01,2025-01-01 08:01:00,2025-01-01 08:01:00,8.984920,77.60124,1,0.0,8.350,8.35,8.35,...,997,7760,2025-01-01,0,0.000,8.35,0,0,0.093408,0
1,2025-01-01,2025-01-01 08:01:00,2025-01-01 08:01:00,9.000140,77.61005,1,0.0,2.680,2.68,2.68,...,999,7761,2025-01-01,0,0.000,2.68,0,0,0.106868,0
2,2025-01-01,2025-01-01 08:01:00,2025-01-01 08:01:00,9.278090,78.04165,1,0.0,1.580,1.58,1.58,...,1029,7804,2025-01-01,0,0.000,1.58,0,0,0.096588,0
3,2025-01-01,2025-01-01 19:49:00,2025-01-01 19:49:00,9.562105,77.63444,2,0.0,1.105,1.23,2.21,...,1061,7763,2025-01-01,1,0.615,0.49,0,1,0.005439,0
4,2025-01-01,2025-01-01 07:38:00,2025-01-01 07:38:00,9.630170,76.47209,1,0.0,2.110,2.11,2.11,...,1068,7647,2025-01-01,0,0.000,2.11,0,0,0.108285,0


In [2]:
print("Rows:", len(data))
print("Columns:", len(data.columns))

print("\nData types:")
print(data.dtypes)

print("\nMissing values:")
print(data.isna().sum().sort_values(ascending=False).head(10))

Rows: 570717
Columns: 32

Data types:
acq_date                          str
start_time                        str
end_time                          str
latitude                      float64
longitude                     float64
observation_count               int64
duration_minutes              float64
mean_frp                      float64
peak_frp                      float64
total_frp                     float64
frp_range                     float64
frp_peak_ratio                float64
frp_per_observation           float64
mean_brightness               float64
start_hour                      int64
day_of_week                     int64
month                           int64
is_weekend                      int64
persistent                      int64
long_duration                   int64
is_day                          int64
is_night                        int64
lat_cell                        int64
lon_cell                        int64
date_only                         str
historical_d

In [3]:
print("Duplicate complete rows:", data.duplicated().sum())

Duplicate complete rows: 0


Check target/anomaly distribution

In [4]:
print("Anomaly distribution:")
print(data["is_anomaly"].value_counts())

print("\nPercentage:")
print(
    data["is_anomaly"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Anomaly distribution:
is_anomaly
0    446255
1    124462
Name: count, dtype: int64

Percentage:
is_anomaly
0    78.19
1    21.81
Name: proportion, dtype: float64


Identify ML feature columns

In [5]:
feature_columns = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night",
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "local_frp_zscore",
    "historical_daily_activity",
    "previously_detected"
]

print("Number of candidate features:", len(feature_columns))

for i, feature in enumerate(feature_columns, 1):
    print(f"{i:02d}. {feature}")

Number of candidate features: 23
01. observation_count
02. duration_minutes
03. mean_frp
04. peak_frp
05. total_frp
06. frp_range
07. frp_peak_ratio
08. frp_per_observation
09. mean_brightness
10. start_hour
11. day_of_week
12. month
13. is_weekend
14. persistent
15. long_duration
16. is_day
17. is_night
18. historical_detection_count
19. historical_mean_frp
20. local_frp_deviation
21. local_frp_zscore
22. historical_daily_activity
23. previously_detected


Check feature availability

In [9]:
# ============================================================
# CELL 5 — CHECK AVAILABLE FEATURES
# ============================================================

print("All columns in dataset:\n")

for i, col in enumerate(data.columns, 1):
    print(f"{i:02d}. {col}")

All columns in dataset:

01. acq_date
02. start_time
03. end_time
04. latitude
05. longitude
06. observation_count
07. duration_minutes
08. mean_frp
09. peak_frp
10. total_frp
11. frp_range
12. frp_peak_ratio
13. frp_per_observation
14. mean_brightness
15. start_hour
16. day_of_week
17. month
18. is_weekend
19. persistent
20. long_duration
21. is_day
22. is_night
23. lat_cell
24. lon_cell
25. date_only
26. historical_detection_count
27. historical_mean_frp
28. local_frp_deviation
29. historical_daily_activity
30. previously_detected
31. anomaly_score
32. is_anomaly


In [10]:
# ============================================================
# CELL 6 — IDENTIFY CONTEXT FEATURES
# ============================================================

context_keywords = [
    "historical",
    "local",
    "regional",
    "previous",
    "activity"
]

context_columns = [
    col for col in data.columns
    if any(keyword in col.lower() for keyword in context_keywords)
]

print("Context-related columns:")
for col in context_columns:
    print("-", col)

Context-related columns:
- historical_detection_count
- historical_mean_frp
- local_frp_deviation
- historical_daily_activity
- previously_detected


In [11]:
# ============================================================
# CELL 7 — SAFE FEATURE LIST
# ============================================================

candidate_features = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night",
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "local_frp_zscore",
    "historical_daily_activity",
    "previously_detected"
]

feature_columns = [
    col for col in candidate_features
    if col in data.columns
]

missing_features = [
    col for col in candidate_features
    if col not in data.columns
]

print("Available features:", len(feature_columns))
print(feature_columns)

print("\nNot available:")
print(missing_features)

Available features: 22
['observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'historical_detection_count', 'historical_mean_frp', 'local_frp_deviation', 'historical_daily_activity', 'previously_detected']

Not available:
['local_frp_zscore']


In [12]:
# infinite value check

# ============================================================
# CELL 8 — INFINITE VALUE CHECK
# ============================================================

X_check = data[feature_columns].copy()

inf_counts = np.isinf(X_check).sum()

print("Total infinite values:", inf_counts.sum())

if inf_counts.sum() > 0:
    print("\nColumns containing infinite values:")
    print(inf_counts[inf_counts > 0])
else:
    print("✅ No infinite values found.")

Total infinite values: 0
✅ No infinite values found.


feature statistics

In [13]:
# ============================================================
# CELL 9 — FEATURE STATISTICS
# ============================================================

feature_summary = data[feature_columns].describe().T

print("Feature statistics:")
display(
    feature_summary[
        ["count", "mean", "std", "min", "50%", "max"]
    ]
)

Feature statistics:


,count,mean,std,min,50%,max
observation_count,570717.0,1.688886,1.407406,1.000000e+00,1.000000,26.000000
duration_minutes,570717.0,6.613134,13.435065,0.000000e+00,0.000000,100.000000
mean_frp,570717.0,7.322063,17.803542,0.000000e+00,3.895000,2056.400000
peak_frp,570717.0,8.979743,26.284301,0.000000e+00,4.130000,2056.400000
total_frp,570717.0,15.249866,66.501370,0.000000e+00,5.030000,4853.380000
frp_range,570717.0,1.657679,12.950197,-7.105427e-15,0.000000,1574.880000
frp_peak_ratio,570682.0,1.115898,0.275084,1.000000e+00,1.000000,8.414053
frp_per_observation,570717.0,7.322063,17.803542,0.000000e+00,3.895000,2056.400000
mean_brightness,570717.0,329.191767,16.554735,2.073800e+02,334.290000,495.100000
start_hour,570717.0,10.828912,5.654838,2.000000e+00,8.000000,22.000000


correlation check

In [14]:
# ============================================================
# CELL 10 — HIGH CORRELATION CHECK
# ============================================================

corr = data[feature_columns].corr()

high_corr = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        value = corr.iloc[i, j]

        if abs(value) >= 0.90:
            high_corr.append(
                (
                    corr.columns[i],
                    corr.columns[j],
                    round(value, 3)
                )
            )

if high_corr:
    print("Highly correlated feature pairs (|r| >= 0.90):\n")

    for feature1, feature2, correlation in high_corr:
        print(
            f"{feature1:<30} "
            f"{feature2:<30} "
            f"r = {correlation}"
        )
else:
    print("✅ No feature pairs with |r| >= 0.90")

Highly correlated feature pairs (|r| >= 0.90):

mean_frp                       frp_per_observation            r = 1.0
start_hour                     is_day                         r = -0.987
start_hour                     is_night                       r = 0.987
is_day                         is_night                       r = -1.0
historical_detection_count     historical_daily_activity      r = 1.0


temporal coverage

In [17]:
# ============================================================
# CELL 11 — TEMPORAL COVERAGE
# ============================================================

data["acq_date"] = pd.to_datetime(
    data["acq_date"],
    errors="coerce"
)

data["start_time"] = pd.to_datetime(
    data["start_time"],
    errors="coerce"
)

data["end_time"] = pd.to_datetime(
    data["end_time"],
    errors="coerce"
)

print("Invalid acq_date:", data["acq_date"].isna().sum())
print("Invalid start_time:", data["start_time"].isna().sum())
print("Invalid end_time:", data["end_time"].isna().sum())

print("\nStart date:", data["acq_date"].min())
print("End date:", data["acq_date"].max())

print("\nEvents by month:")

monthly_counts = (
    data["acq_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

print(monthly_counts)

Invalid acq_date: 0
Invalid start_time: 0
Invalid end_time: 0

Start date: 2025-01-01 00:00:00
End date: 2025-03-19 00:00:00

Events by month:
acq_date
2025-01    107774
2025-02    168923
2025-03    294020
Freq: M, Name: count, dtype: int64


In [18]:
# ============================================================
# CELL 12 — SPATIAL COVERAGE
# ============================================================

print("Latitude range:")
print(data["latitude"].min(), "→", data["latitude"].max())

print("\nLongitude range:")
print(data["longitude"].min(), "→", data["longitude"].max())

print("\nUnique coordinates:")
print(
    data[["latitude", "longitude"]]
    .drop_duplicates()
    .shape[0]
)

Latitude range:
8.07834 → 34.49413

Longitude range:
68.5089 → 96.98654

Unique coordinates:
439244


In [19]:
# ============================================================
# CELL 13 — FINAL VALIDATION SUMMARY
# ============================================================

validation_summary = {
    "rows": int(len(data)),
    "columns": int(len(data.columns)),
    "candidate_features": int(len(feature_columns)),
    "missing_feature_values": int(
        data[feature_columns].isna().sum().sum()
    ),
    "infinite_values": int(
        np.isinf(data[feature_columns]).sum().sum()
    ),
    "duplicate_rows": int(data.duplicated().sum()),
    "start_date": str(data["acq_date"].min().date()),
    "end_date": str(data["acq_date"].max().date()),
}

for key, value in validation_summary.items():
    print(f"{key}: {value}")

rows: 570717
columns: 32
candidate_features: 22
missing_feature_values: 35
infinite_values: 0
duplicate_rows: 0
start_date: 2025-01-01
end_date: 2025-03-19


In [20]:
# ============================================================
# CELL 14 — FIX MISSING FEATURE VALUES
# ============================================================

print("Missing values before:")
print(
    data[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Fix FRP ratio
data["frp_peak_ratio"] = (
    data["frp_peak_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("\nMissing values after:")
print(
    data[feature_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(
    "\nTotal missing feature values:",
    data[feature_columns].isna().sum().sum()
)

Missing values before:
frp_peak_ratio         35
observation_count       0
mean_frp                0
peak_frp                0
total_frp               0
duration_minutes        0
frp_range               0
frp_per_observation     0
mean_brightness         0
start_hour              0
dtype: int64

Missing values after:
observation_count      0
duration_minutes       0
mean_frp               0
peak_frp               0
total_frp              0
frp_range              0
frp_peak_ratio         0
frp_per_observation    0
mean_brightness        0
start_hour             0
dtype: int64

Total missing feature values: 0


Save the corrected ML dataset

In [21]:
# ============================================================
# CELL 15 — SAVE CORRECTED DATASET
# ============================================================

ML_READY_DIR = PROJECT_ROOT / "data" / "ml_ready"
ML_READY_DIR.mkdir(parents=True, exist_ok=True)

ML_FILE_CORRECTED = (
    ML_READY_DIR /
    "thermal_context_ml_ready_prototype_v2.csv"
)

data.to_csv(
    ML_FILE_CORRECTED,
    index=False
)

print("Saved corrected dataset:")
print(ML_FILE_CORRECTED)

print("\nShape:", data.shape)

Saved corrected dataset:
C:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\ml_ready\thermal_context_ml_ready_prototype_v2.csv

Shape: (570717, 32)


Final validation

In [22]:
# ============================================================
# CELL 16 — FINAL VALIDATION
# ============================================================

print("Rows:", len(data))
print("Columns:", len(data.columns))

print("Missing feature values:",
      data[feature_columns].isna().sum().sum())

print("Infinite values:",
      np.isinf(data[feature_columns]).sum().sum())

print("Duplicate rows:",
      data.duplicated().sum())

print("\nAnomaly distribution:")
print(
    data["is_anomaly"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Rows: 570717
Columns: 32
Missing feature values: 0
Infinite values: 0
Duplicate rows: 0

Anomaly distribution:
is_anomaly
0    78.19
1    21.81
Name: proportion, dtype: float64


Sort chronologically


In [23]:
# ============================================================
# CELL 17 — CHRONOLOGICAL ORDER
# ============================================================

data = data.sort_values(
    ["start_time", "latitude", "longitude"]
).reset_index(drop=True)

print("Chronologically sorted:", data["start_time"].is_monotonic_increasing)

print("First event:")
print(data["start_time"].iloc[0])

print("\nLast event:")
print(data["start_time"].iloc[-1])

Chronologically sorted: True
First event:
2025-01-01 03:50:00

Last event:
2025-03-19 21:30:00


Check historical feature behavior

In [24]:
# ============================================================
# CELL 18 — HISTORICAL FEATURE CHECK
# ============================================================

historical_features = [
    "historical_detection_count",
    "historical_mean_frp",
    "local_frp_deviation",
    "historical_daily_activity",
    "previously_detected"
]

print("Historical feature statistics:\n")

display(
    data[historical_features].describe().T[
        ["count", "mean", "std", "min", "50%", "max"]
    ]
)

Historical feature statistics:



,count,mean,std,min,50%,max
historical_detection_count,570717.0,26.231600,97.412819,0.00,2.000000,1571.000000
historical_mean_frp,570717.0,4.703545,13.821294,0.00,2.035000,1121.475833
local_frp_deviation,570717.0,2.565792,12.762247,-680.01,1.527009,2056.400000
historical_daily_activity,570717.0,24.357026,95.899743,0.00,0.000000,1551.000000
previously_detected,570717.0,0.696380,0.459821,0.00,1.000000,1.000000


Check first observations

In [25]:
# ============================================================
# CELL 19 — FIRST EVENTS CHECK
# ============================================================

first_events = data[
    [
        "start_time",
        "latitude",
        "longitude",
        "historical_detection_count",
        "historical_mean_frp",
        "local_frp_deviation",
        "historical_daily_activity",
        "previously_detected"
    ]
].head(20)

display(first_events)

,start_time,latitude,longitude,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected
0,2025-01-01 03:50:00,26.9361,94.0335,0,0.000000,6.500000,0,0
1,2025-01-01 03:51:00,25.3231,91.6008,0,0.000000,8.000000,0,0
2,2025-01-01 03:51:00,25.3249,91.5900,4,7.630333,0.909667,0,1
3,2025-01-01 03:51:00,25.3249,91.5900,4,7.630333,0.909667,0,1
4,2025-01-01 03:51:00,25.3249,91.5900,4,7.630333,0.909667,0,1
5,2025-01-01 03:51:00,26.2760,92.8154,0,0.000000,11.600000,0,0
6,2025-01-01 03:51:00,26.7587,92.7613,0,0.000000,5.300000,0,0
7,2025-01-01 03:52:00,20.9661,85.1728,13,4.146439,-0.718581,0,1
8,2025-01-01 03:52:00,20.9661,85.1728,13,4.146439,-0.718581,0,1
9,2025-01-01 03:52:00,20.9661,85.1728,13,4.146439,-0.718581,0,1


Check temporal split

In [26]:
# ============================================================
# CELL 20 — MONTHLY DISTRIBUTION
# ============================================================

data["month_period"] = data["start_time"].dt.to_period("M")

monthly = (
    data["month_period"]
    .value_counts()
    .sort_index()
)

print(monthly)

month_period
2025-01    107774
2025-02    168923
2025-03    294020
Freq: M, Name: count, dtype: int64


Create temporal split

In [27]:
# ============================================================
# CELL 21 — TIME-BASED TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train = data[
    data["start_time"] < "2025-02-01"
].copy()

validation = data[
    (data["start_time"] >= "2025-02-01") &
    (data["start_time"] < "2025-03-01")
].copy()

test = data[
    data["start_time"] >= "2025-03-01"
].copy()

print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

print("\nTotal:", len(train) + len(validation) + len(test))

Train: 107774
Validation: 168923
Test: 294020

Total: 570717


In [28]:
# ============================================================
# CELL 22 — VERIFY SPLIT DATES
# ============================================================

for name, subset in [
    ("TRAIN", train),
    ("VALIDATION", validation),
    ("TEST", test)
]:
    print(f"\n{name}")
    print("Start:", subset["start_time"].min())
    print("End  :", subset["start_time"].max())


TRAIN
Start: 2025-01-01 03:50:00
End  : 2025-01-31 21:13:00

VALIDATION
Start: 2025-02-01 04:43:00
End  : 2025-02-28 21:37:00

TEST
Start: 2025-03-01 03:37:00
End  : 2025-03-19 21:30:00


In [29]:
# ============================================================
# CELL 25 — CHECK SAME-TIME SAME-LOCATION EVENTS
# ============================================================

same_location_time = (
    data.groupby(
        ["start_time", "latitude", "longitude"]
    )
    .size()
    .reset_index(name="count")
)

print("Total unique time-location groups:",
      len(same_location_time))

print(
    "Groups containing multiple rows:",
    (same_location_time["count"] > 1).sum()
)

print("\nLargest group sizes:")
print(
    same_location_time["count"]
    .value_counts()
    .sort_index()
    .tail(10)
)

Total unique time-location groups: 439251
Groups containing multiple rows: 105125

Largest group sizes:
count
1    334126
2     85222
3     14784
4      3920
5      1095
6        96
8         8
Name: count, dtype: int64


In [30]:
# ============================================================
# CELL 25 — CHECK SAME-TIME SAME-LOCATION EVENTS
# ============================================================

same_location_time = (
    data.groupby(
        ["start_time", "latitude", "longitude"]
    )
    .size()
    .reset_index(name="count")
)

print("Total unique time-location groups:",
      len(same_location_time))

print(
    "Groups containing multiple rows:",
    (same_location_time["count"] > 1).sum()
)

print("\nLargest group sizes:")
print(
    same_location_time["count"]
    .value_counts()
    .sort_index()
    .tail(10)
)

Total unique time-location groups: 439251
Groups containing multiple rows: 105125

Largest group sizes:
count
1    334126
2     85222
3     14784
4      3920
5      1095
6        96
8         8
Name: count, dtype: int64


In [32]:
# ============================================================
# CELL 25-27 — CHECK SAME-TIME SAME-LOCATION GROUPS
# ============================================================

# 1. Group events by exact start time + location
same_location_time = (
    data.groupby(
        ["start_time", "latitude", "longitude"]
    )
    .size()
    .reset_index(name="count")
)

print("Total rows:", len(data))
print("Unique time-location groups:", len(same_location_time))

multi_groups = same_location_time[
    same_location_time["count"] > 1
].sort_values(
    "count",
    ascending=False
)

print(
    "Groups containing multiple rows:",
    len(multi_groups)
)

print("\nLargest group sizes:")
print(
    same_location_time["count"]
    .value_counts()
    .sort_index()
    .tail(10)
)

print("\nTop 20 multi-row groups:")
display(multi_groups.head(20))


# 2. Inspect the largest group
if len(multi_groups) > 0:

    example = multi_groups.iloc[0]

    print("\nLargest group:")
    print("Start time:", example["start_time"])
    print("Latitude:", example["latitude"])
    print("Longitude:", example["longitude"])
    print("Rows:", example["count"])

    example_rows = data[
        (data["start_time"] == example["start_time"]) &
        (data["latitude"] == example["latitude"]) &
        (data["longitude"] == example["longitude"])
    ].copy()

    print("\nRows belonging to this group:")
    display(example_rows)

else:
    print("\n✅ No multiple rows at the same exact time and location.")

Total rows: 570717
Unique time-location groups: 439251
Groups containing multiple rows: 105125

Largest group sizes:
count
1    334126
2     85222
3     14784
4      3920
5      1095
6        96
8         8
Name: count, dtype: int64

Top 20 multi-row groups:


,start_time,latitude,longitude,count
384927,2025-03-15 19:23:00,22.043405,83.735230,8
384583,2025-03-15 15:36:00,22.044100,83.734800,8
378642,2025-03-15 07:48:00,22.042280,83.733140,8
374447,2025-03-15 06:55:00,22.040580,83.736780,8
384371,2025-03-15 09:13:00,22.038400,83.730100,8
372753,2025-03-15 04:42:00,22.043200,83.734200,8
386330,2025-03-15 20:17:00,22.041649,83.734508,8
387439,2025-03-15 21:20:00,22.043600,83.739700,8
23247,2025-01-09 06:51:00,22.042745,83.733852,6
416028,2025-03-17 21:00:00,22.042800,83.735300,6



Largest group:
Start time: 2025-03-15 19:23:00
Latitude: 22.043405
Longitude: 83.73523
Rows: 8

Rows belonging to this group:


,acq_date,start_time,end_time,latitude,longitude,observation_count,duration_minutes,mean_frp,peak_frp,total_frp,...,lon_cell,date_only,historical_detection_count,historical_mean_frp,local_frp_deviation,historical_daily_activity,previously_detected,anomaly_score,is_anomaly,month_period
499527,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.111132,1,2025-03
499528,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,0.089818,0,2025-03
499529,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.036089,1,2025-03
499530,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.074705,1,2025-03
499531,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.121998,1,2025-03
499532,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.062100,1,2025-03
499533,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.185409,1,2025-03
499534,2025-03-15,2025-03-15 19:23:00,2025-03-15 19:23:00,22.043405,83.73523,2,0.0,4.035,4.24,8.07,...,8373,2025-03-15,1508,9.976889,4.521006,1490,1,-0.122912,1,2025-03


In [33]:
# ============================================================
# CHECK ANOMALY TABLE
# ============================================================

ANOMALY_FILE = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "thermal_anomaly_prototype.csv"
)

anomaly = pd.read_csv(ANOMALY_FILE)

print("Anomaly table shape:", anomaly.shape)

print("\nColumns:")
print(anomaly.columns.tolist())

print("\nDuplicate rows:",
      anomaly.duplicated().sum())

Anomaly table shape: (439251, 25)

Columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']

Duplicate rows: 0


In [34]:
# ============================================================
# CHECK ANOMALY KEY UNIQUENESS
# ============================================================

anomaly_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

print(
    "Total anomaly rows:",
    len(anomaly)
)

print(
    "Unique anomaly keys:",
    anomaly[anomaly_key].drop_duplicates().shape[0]
)

print(
    "Duplicate anomaly keys:",
    anomaly.duplicated(
        subset=anomaly_key
    ).sum()
)

Total anomaly rows: 439251
Unique anomaly keys: 439251
Duplicate anomaly keys: 0


In [35]:
# ============================================================
# CHECK THERMAL EVENT TABLE
# ============================================================

EVENT_FILE = (
    PROJECT_ROOT
    / "data"
    / "events"
    / "thermal_events_prototype.csv"
)

events = pd.read_csv(EVENT_FILE)

print("Thermal events shape:", events.shape)

print(
    "Unique event keys:",
    events[anomaly_key].drop_duplicates().shape[0]
)

print(
    "Duplicate event keys:",
    events.duplicated(
        subset=anomaly_key
    ).sum()
)

Thermal events shape: (439251, 14)
Unique event keys: 439251
Duplicate event keys: 0


In [37]:
# ============================================================
# CHECK ANOMALY TABLE
# ============================================================

ANOMALY_FILE = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "thermal_anomaly_prototype.csv"
)

anomaly = pd.read_csv(ANOMALY_FILE)

print("Anomaly table shape:", anomaly.shape)

print("\nColumns:")
print(anomaly.columns.tolist())

print("\nDuplicate complete rows:",
      anomaly.duplicated().sum())

Anomaly table shape: (439251, 25)

Columns:
['acq_date', 'start_time', 'end_time', 'latitude', 'longitude', 'observation_count', 'duration_minutes', 'mean_frp', 'peak_frp', 'total_frp', 'frp_range', 'frp_peak_ratio', 'frp_per_observation', 'mean_brightness', 'start_hour', 'day_of_week', 'month', 'is_weekend', 'persistent', 'long_duration', 'is_day', 'is_night', 'anomaly_score', 'anomaly_prediction', 'is_anomaly']

Duplicate complete rows: 0


In [38]:
# ============================================================
# CHECK ANOMALY EVENT KEY
# ============================================================

anomaly_key = [
    "acq_date",
    "start_time",
    "end_time",
    "latitude",
    "longitude"
]

print("Total anomaly rows:", len(anomaly))

print(
    "Unique anomaly event keys:",
    anomaly[anomaly_key].drop_duplicates().shape[0]
)

print(
    "Duplicate anomaly event keys:",
    anomaly.duplicated(
        subset=anomaly_key
    ).sum()
)

Total anomaly rows: 439251
Unique anomaly event keys: 439251
Duplicate anomaly event keys: 0
